In [1]:
import numpy as np
import pymc as pm
import arviz as az

In [2]:
np.random.seed(1)

#### Competing Bayesian Regression Models

In [4]:
# Simulating data.

n = 300

x1 = np.random.normal(0, 1, n)
x2 = np.random.normal(0, 1, n)

true_intercept = -0.5
true_beta1 = 2.0
true_beta2 = 1.0

linear = (
    true_intercept
    + true_beta1 * x1
    + true_beta2 * x2
)

p = 1 / (1 + np.exp(-linear))

y = np.random.binomial(1, p)

print("Sample size:", n)

Sample size: 300


In [5]:
# Model 1: Simple logistic regression.

with pm.Model() as simple_model:

    intercept = pm.Normal(
        "intercept",
        mu=0,
        sigma=5
    )

    beta1 = pm.Normal(
        "beta1",
        mu=0,
        sigma=5
    )

    probability = pm.math.sigmoid(
        intercept + beta1 * x1
    )

    outcome = pm.Bernoulli(
        "outcome",
        p=probability,
        observed=y
    )

    simple_trace = pm.sample(
        draws=2000,
        tune=1000,
        random_seed=1,
        return_inferencedata=True
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, beta1]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 45 seconds.


In [6]:
# Model 2: More complex logistic regression.

with pm.Model() as complex_model:

    intercept = pm.Normal(
        "intercept",
        mu=0,
        sigma=5
    )

    beta1 = pm.Normal(
        "beta1",
        mu=0,
        sigma=5
    )

    beta2 = pm.Normal(
        "beta2",
        mu=0,
        sigma=5
    )

    probability = pm.math.sigmoid(
        intercept
        + beta1 * x1
        + beta2 * x2
    )

    outcome = pm.Bernoulli(
        "outcome",
        p=probability,
        observed=y
    )

    complex_trace = pm.sample(
        draws=2000,
        tune=1000,
        random_seed=1,
        return_inferencedata=True
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, beta1, beta2]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 47 seconds.


#### Compare Models Using LOO and WAIC

In [7]:
with simple_model:
    simple_trace = pm.compute_log_likelihood(simple_trace)

with complex_model:
    complex_trace = pm.compute_log_likelihood(complex_trace)

Output()

Output()

In [8]:
# Computing LOO.

simple_loo = az.loo(simple_trace)

complex_loo = az.loo(complex_trace)

print("Simple model:\n", simple_loo)

print("\nComplex model:\n", complex_loo)

Simple model:
 Computed from 8000 posterior samples and 300 observations log-likelihood matrix.

         Estimate       SE
elpd_loo  -156.05     8.55
p_loo        1.98        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.70]   (good)      300  100.0%
   (0.70, 1]   (bad)         0    0.0%
   (1, Inf)   (very bad)    0    0.0%


Complex model:
 Computed from 8000 posterior samples and 300 observations log-likelihood matrix.

         Estimate       SE
elpd_loo  -128.74     9.51
p_loo        2.98        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.70]   (good)      300  100.0%
   (0.70, 1]   (bad)         0    0.0%
   (1, Inf)   (very bad)    0    0.0%



In [9]:
# Computing WAIC.

simple_waic = az.waic(simple_trace)

complex_waic = az.waic(complex_trace)

print("Simple model:\n", simple_waic)

print("\nComplex model:\n", complex_waic)

Simple model:
 Computed from 8000 posterior samples and 300 observations log-likelihood matrix.

          Estimate       SE
elpd_waic  -156.05     8.55
p_waic        1.98        -

Complex model:
 Computed from 8000 posterior samples and 300 observations log-likelihood matrix.

          Estimate       SE
elpd_waic  -128.73     9.51
p_waic        2.98        -


In [10]:
# Comparing models.

comparison = az.compare(
    {"Simple Model": simple_trace, "Complex Model": complex_trace},
    ic="loo"
)

print(comparison)

               rank    elpd_loo     p_loo  elpd_diff    weight        se  \
Complex Model     0 -128.735823  2.981223   0.000000  0.963619  9.512378   
Simple Model      1 -156.050537  1.983696  27.314714  0.036381  8.549332   

                    dse  warning scale  
Complex Model  0.000000    False   log  
Simple Model   7.327887    False   log  


In [11]:
# Interpreting the results.

best_model = comparison.index[0]

print("Preferred Model: ", best_model)

if best_model == "Simple Model":
    print("The simpler model has better expected predictive performance.")
else:
    print("The more complex model has better expected predictive performance.")

Preferred Model:  Complex Model
The more complex model has better expected predictive performance.
